<a href="https://colab.research.google.com/github/pop123-ux/HuggingFace-Project-Learning/blob/main/course/chapter6/Fast_Tokenizers_Special_Powers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fast tokenizers' special powers (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
!pip install datasets evaluate transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.1 MB/s eta 0:00:00


In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
example = "My name is Alex and I am currently learning the Hugging Face API"
encoding = tokenizer(example)
print(type(encoding))

<class 'transformers.tokenization_utils_base.BatchEncoding'>


Since the AutoTokenizer class picks a fast tokenizer by default, we can use the additional methods this BatchEncoding object provides.

We have two ways to check if our tokenize is a fast or slow one.

We can either check the attribute is_fast of the tokenizer:

In [7]:
tokenizer.is_fast

True

or check the same attribute on our encoding:

In [8]:
encoding.is_fast

True

Let's see what a fast tokenizer enables us to do. First, we can access the tokens without having to convert the IDs back to tokens:

In [9]:
encoding.tokens()

['[CLS]',
 'My',
 'name',
 'is',
 'Alex',
 'and',
 'I',
 'am',
 'currently',
 'learning',
 'the',
 'Hu',
 '##gging',
 'Face',
 'API',
 '[SEP]']

As we can see right here, different tokenizers tokenize the sequences differently. Hence, the following demonstration with the roberta-base and bert-base-cased checkpoints:

In [12]:
from transformers import AutoTokenizer

tokenizer1 = AutoTokenizer.from_pretrained("roberta-base")
tokenizer2 = AutoTokenizer.from_pretrained("bert-base-cased")
example2 = "81s"
encoding1 = tokenizer1(example2)
encoding2 = tokenizer2(example2)
encoding1.tokens(), encoding2.tokens()

(['<s>', '81', 's', '</s>'], ['[CLS]', '81', '##s', '[SEP]'])

We can map any word or token to characters in the original text, and vice versa, via the words_to_chars() or token_to_chars() and char_to_word() or char_to_token() methods.

For instance, the word_ids() method told us that ##gging is part of the word at index 10, but which word is it in the sentence?

We can find out like this:

In [21]:
start, end = encoding.word_to_chars(10)
example[start:end]

'Hugging'

This is all powered by the fact that the fast tokenizer keeps track of the span of text each token comes from in a list of offsets.

## Getting the base results with the pipeline ##

First, let's grab a token classification pipeline so we can get some results to compare manually.

The model used by default is dbmdz/bert-large-cased-finetuned-conll03-english; it performs NER on sentences:

In [23]:
from transformers import pipeline

token_classifier = pipeline("token-classification", aggregation_strategy='simple')
token_classifier("My name is Alex and I am currently learning the Hugging Face API.")

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity_group': 'PER',
  'score': np.float32(0.9983621),
  'word': 'Alex',
  'start': 11,
  'end': 15},
 {'entity_group': 'MISC',
  'score': np.float32(0.9406351),
  'word': 'Hugging Face API',
  'start': 48,
  'end': 64}]

## From inputs to predictions ##

First we need to tokenize our input and pass it through the model.

We instantiate the tokenizer and the model using the AutoXxx classes and then we use them on our example:

In [24]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_checkpoint = "dbmdz/bert-large-cased-finetuned-conll03-english"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint)

example = "My name is Alex and I am currently learning the Hugging Face API."
inputs = tokenizer(example, return_tensors='pt')
outputs = model(**inputs)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [25]:
inputs['input_ids'].shape, outputs.logits.shape

(torch.Size([1, 17]), torch.Size([1, 17, 9]))

We have a batch with 1 sequence of 17 tokens and the model has 9 different labels, so the output of the model has a shape of 1 x 17 x 9.

Like for the text classification pipeline, we use a softmax function to convert those logits to probabilities, and we take the argmax to get predictions

In [26]:
import torch

probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)[0].tolist()
predictions = outputs.logits.argmax(dim=-1)[0].tolist()
probabilities, predictions

([[0.9998061060905457,
   1.1843117135867942e-05,
   4.93016850668937e-05,
   1.0378174010838848e-05,
   1.603488090040628e-05,
   9.932337889040355e-06,
   1.6381451132474467e-05,
   1.1457143045845442e-05,
   6.856261461507529e-05],
  [0.9996719360351562,
   6.229599421203602e-06,
   2.9680273655685596e-05,
   4.929908755002543e-06,
   0.00014253082918003201,
   8.511703526892234e-06,
   7.794768316671252e-05,
   8.283560418931302e-06,
   4.996521965949796e-05],
  [0.9998651742935181,
   4.512535724643385e-06,
   1.5718786016805097e-05,
   3.3855073979793815e-06,
   3.955521606258117e-05,
   4.4385396904544905e-06,
   2.9429833375616e-05,
   5.832447641296312e-06,
   3.1917006708681583e-05],
  [0.9999327659606934,
   2.550049430283252e-06,
   7.171980087150587e-06,
   2.464143562974641e-06,
   1.7217669665114954e-05,
   3.1772729016665835e-06,
   1.8386093870503828e-05,
   3.6703379464597674e-06,
   1.2633204278245103e-05],
  [0.00032588286558166146,
   4.3098032620036975e-05,
   0.0

The model.config.id2label attribute contains the mapping of indexes to labels that we can use to make sense of the predictions:

In [27]:
model.config.id2label

{0: 'O',
 1: 'B-MISC',
 2: 'I-MISC',
 3: 'B-PER',
 4: 'I-PER',
 5: 'B-ORG',
 6: 'I-ORG',
 7: 'B-LOC',
 8: 'I-LOC'}

Next, we grab the score and label of each token that was not classified as 0:

In [29]:
results = []
tokens = inputs.tokens()

for idx, pred in enumerate(predictions):
  label = model.config.id2label[pred]
  if label != "O":
    results.append(
        {'entity': label, 'score': probabilities[idx][pred], 'word': tokens[idx]}
    )

inputs.tokens(), results

(['[CLS]',
  'My',
  'name',
  'is',
  'Alex',
  'and',
  'I',
  'am',
  'currently',
  'learning',
  'the',
  'Hu',
  '##gging',
  'Face',
  'API',
  '.',
  '[SEP]'],
 [{'entity': 'I-PER', 'score': 0.998362123966217, 'word': 'Alex'},
  {'entity': 'I-MISC', 'score': 0.9777467846870422, 'word': 'Hu'},
  {'entity': 'I-MISC', 'score': 0.829853892326355, 'word': '##gging'},
  {'entity': 'I-MISC', 'score': 0.9789395928382874, 'word': 'Face'},
  {'entity': 'I-MISC', 'score': 0.9760001301765442, 'word': 'API'}])

The pipeline also gave us information about the start and end of each entity in the original sentence.

This is where our offset mapping will come into play. To get the offsets, we just have to set return_offsets_mapping=True when we apply the tokenizer to our inputs:

In [30]:
inputs_with_offsets = tokenizer(example, return_offsets_mapping=True)
inputs_with_offsets['offset_mapping']

[(0, 0),
 (0, 2),
 (3, 7),
 (8, 10),
 (11, 15),
 (16, 19),
 (20, 21),
 (22, 24),
 (25, 34),
 (35, 43),
 (44, 47),
 (48, 50),
 (50, 55),
 (56, 60),
 (61, 64),
 (64, 65),
 (0, 0)]

Each tuple is the span of text corresponding to each token, where (0, 0) is reserved for the special tokens.

In [35]:
example[50:55]

'gging'

As we can see we get the proper span of text without the ##:

'gging'

Using this we can now complete the previous results:

In [36]:
results = []
inputs_with_offsets = tokenizer(example, return_offsets_mapping=True)
tokens = inputs_with_offsets.tokens()
offsets = inputs_with_offsets['offset_mapping']

for idx, pred in enumerate(predictions):
  label = model.config.id2label[pred]
  if label != "O":
    start, end = offsets[idx]
    results.append(
        {
            "entity": label,
            "score": probabilities[idx][pred],
            "word": tokens[idx],
            "start": start,
            "end": end
        }
    )

results

[{'entity': 'I-PER',
  'score': 0.998362123966217,
  'word': 'Alex',
  'start': 11,
  'end': 15},
 {'entity': 'I-MISC',
  'score': 0.9777467846870422,
  'word': 'Hu',
  'start': 48,
  'end': 50},
 {'entity': 'I-MISC',
  'score': 0.829853892326355,
  'word': '##gging',
  'start': 50,
  'end': 55},
 {'entity': 'I-MISC',
  'score': 0.9789395928382874,
  'word': 'Face',
  'start': 56,
  'end': 60},
 {'entity': 'I-MISC',
  'score': 0.9760001301765442,
  'word': 'API',
  'start': 61,
  'end': 64}]

This is the same as what we got from the first pipeline!

## Grouping entities ##

using the offset to determine the start and end keys for each entity is handy, but that information isn't strictly necessary.

When we want to group the entities together, however, the offsets will save us a lot of messy code.

For example, if we wanted to group together the tokens Hu, ##gging, and Face, we could make special rules that say the first two should be attached while removing the ##, and the face should be added with a space since it does not begin with ## - but that would only work for this particular type of tokenizer.

We would have to write another set of rules for a SentencePiece or a Byte-Fair-Encoding tokenizer

In [37]:
import numpy as np

results = []
inputs_with_offsets = tokenizer(example, return_offsets_mapping=True)
tokens = inputs_with_offsets.tokens()
offsets = inputs_with_offsets["offset_mapping"]

idx = 0
while idx < len(predictions):
    pred = predictions[idx]
    label = model.config.id2label[pred]
    if label != "O":
        # Remove the B- or I-
        label = label[2:]
        start, _ = offsets[idx]

        # Grab all the tokens labeled with I-label
        all_scores = []
        while (
            idx < len(predictions)
            and model.config.id2label[predictions[idx]] == f"I-{label}"
        ):
            all_scores.append(probabilities[idx][pred])
            _, end = offsets[idx]
            idx += 1

        # The score is the mean of all the scores of the tokens in that grouped entity
        score = np.mean(all_scores).item()
        word = example[start:end]
        results.append(
            {
                "entity_group": label,
                "score": score,
                "word": word,
                "start": start,
                "end": end,
            }
        )
    idx += 1

print(results)

[{'entity_group': 'PER', 'score': 0.998362123966217, 'word': 'Alex', 'start': 11, 'end': 15}, {'entity_group': 'MISC', 'score': 0.9406351000070572, 'word': 'Hugging Face API', 'start': 48, 'end': 64}]
